# Modal single-GPU overfitting pilot

This is the preregistered Milestone 2 learning pilot for the fixed eight-task
ARC set. It trains the complete FLUX transformer while NVARC and the FLUX VAE
remain frozen.

Success is evaluated in three layers:

1. fixed train-probe loss reaches at most 50% of its step-0 value;
2. at least four canonical queries produce a valid, correctly shaped pass@2 grid;
3. at least one canonical query is exact-grid pass@2 (`6/8` is strong and `8/8` complete overfitting).

Prerequisites:

- one Modal H200;
- the writable `diffusion-arc-models` Volume mounted at `/mnt/diffusion-arc-models`;
- the full ARC-AGI-2 Volume setup notebook completed;
- a `GITHUB_TOKEN` Modal Secret;
- `DIFFUSION_ARC_REVISION` optionally set to the exact pushed commit containing this notebook's APIs (default: `main`).

Run top-to-bottom. Each epoch is a separate, manually triggered paid-compute cell.


In [1]:
!nvidia-smi


Tue Aug 25 06:05:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:C5:00.0 Off |                    0 |
| N/A   37C    P0             79W /  700W |       4MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
%uv pip uninstall torchvision


Using Python 3.12.6 environment at: /usr/local
Uninstalled 1 package in 52ms
 - torchvision==0.23.0+cu129
Note: you may need to restart the kernel to use updated packages.


## 1. Immutable experiment configuration and durable report


In [3]:
from __future__ import annotations

import base64
from dataclasses import asdict, replace
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import time

SEED = 167
PROBE_SEED = 1167
GENERATION_SEEDS = (167, 168)
EPOCH_COUNT = 4
STEPS_PER_EPOCH = 208
PROBE_INTERVAL_STEPS = 52
MAX_GRAD_NORM = 1.0
INFERENCE_STEPS = 50
EXPECTED_TRAIN_SEQUENCES = 512
EXPECTED_TRAIN_TARGETS = 1664
EXPECTED_UNSEEN_SEQUENCES = 128
EXPECTED_UNSEEN_TARGETS = 416
EXPECTED_TASKS = 8
MINIMUM_VRAM_GIB = 120.0

REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_REVISION = '306eb87dae1841669451885102aede283bc22bc4'
if not SOURCE_REVISION:
    raise ValueError('DIFFUSION_ARC_REVISION must not be empty')

VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
FLUX_CHECKPOINT = VOLUME_ROOT / 'flux2-klein-base-4b'
NVARC_CHECKPOINT = VOLUME_ROOT / 'nvarc-qwen3-4b-grids15-sft139'
MODEL_MANIFEST = VOLUME_ROOT / 'download-manifest.json'
ARC_DATA_ROOT = VOLUME_ROOT / 'data' / 'raw' / 'arc-agi-2'
ARC_DATA_MANIFEST = VOLUME_ROOT / 'data' / 'arc-agi-2-manifest.json'
RESULTS_ROOT = VOLUME_ROOT / 'results' / 'single-gpu-overfitting-pilot'

required = (
    FLUX_CHECKPOINT,
    NVARC_CHECKPOINT,
    MODEL_MANIFEST,
    ARC_DATA_ROOT,
    ARC_DATA_MANIFEST,
)
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required Volume artifacts: {missing}')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPORT_DIR = RESULTS_ROOT / RUN_ID
REPORT_PATH = REPORT_DIR / 'report.json'
STEPS_PATH = REPORT_DIR / 'optimizer_steps.jsonl'
IMAGES_DIR = REPORT_DIR / 'generated_images'
REPORT_DIR.mkdir(parents=True, exist_ok=False)
IMAGES_DIR.mkdir()

REPORT: dict[str, object] = {
    'schema_version': 1,
    'status': 'INITIALIZING',
    'run_id': RUN_ID,
    'question': (
        'Can the complete FLUX transformer learn from frozen NVARC conditioning '
        'and overfit a fixed eight-task augmented ARC training set?'
    ),
    'protocol': {
        'seed': SEED,
        'probe_seed': PROBE_SEED,
        'generation_seeds': list(GENERATION_SEEDS),
        'epochs': EPOCH_COUNT,
        'steps_per_epoch': STEPS_PER_EPOCH,
        'probe_interval_steps': PROBE_INTERVAL_STEPS,
        'max_grad_norm': MAX_GRAD_NORM,
        'inference_steps': INFERENCE_STEPS,
        'success_criteria': {
            'maximum_final_train_probe_loss_ratio': 0.5,
            'minimum_correct_shape_pass2': 4,
            'minimum_exact_grid_pass2': 1,
            'strong_exact_grid_pass2': 6,
            'complete_exact_grid_pass2': 8,
        },
    },
    'paths': {
        'volume_root': str(VOLUME_ROOT),
        'flux_checkpoint': str(FLUX_CHECKPOINT),
        'nvarc_checkpoint': str(NVARC_CHECKPOINT),
        'arc_data_root': str(ARC_DATA_ROOT),
        'report': str(REPORT_PATH),
        'step_log': str(STEPS_PATH),
    },
    'failures': [],
    'probes': [],
    'generation': [],
    'epochs': [],
}


def save_report() -> None:
    temporary = REPORT_PATH.with_suffix('.json.tmp')
    temporary.write_text(
        json.dumps(REPORT, indent=2, sort_keys=True, allow_nan=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(REPORT_PATH)
    if hasattr(os, 'sync'):
        os.sync()


def record_failure(stage: str, error: BaseException) -> None:
    failure = {
        'stage': stage,
        'type': type(error).__name__,
        'message': str(error),
        'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    failures = REPORT.setdefault('failures', [])
    if isinstance(failures, list):
        failures.append(failure)
    REPORT['status'] = 'FAIL'
    REPORT['failed_stage'] = stage
    save_report()


def run_command(label: str, command: list[str], *, cwd: Path | None = None, env=None) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-40:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


save_report()
print('Run ID:', RUN_ID)
print('Durable report:', REPORT_PATH)


Run ID: 20260825T060523831464Z
Durable report: /mnt/diffusion-arc-models/results/single-gpu-overfitting-pilot/20260825T060523831464Z/report.json


## 2. Fetch, install, and test the exact Diffusion-ARC source


In [4]:
SOURCE_DIR = Path('/tmp') / f'diffusion-arc-overfit-source-{RUN_ID}'
SOURCE_DIR.mkdir(parents=True)
run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
run_command('add origin', ['git', 'remote', 'add', 'origin', REPOSITORY_URL], cwd=SOURCE_DIR)

github_token = os.environ.get('GITHUB_TOKEN', '').strip()
if not github_token:
    error = RuntimeError("Attach a Modal Secret containing GITHUB_TOKEN")
    record_failure('source-authentication', error)
    raise error
credential = base64.b64encode(
    f'x-access-token:{github_token}'.encode('utf-8')
).decode('ascii')
git_environment = os.environ.copy()
git_environment.pop('GITHUB_TOKEN', None)
git_environment.update({
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
})
try:
    run_command(
        'fetch source',
        ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
        cwd=SOURCE_DIR,
        env=git_environment,
    )
    run_command('checkout source', ['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=SOURCE_DIR)
finally:
    git_environment.clear()
    del git_environment
    del credential

RESOLVED_SOURCE_COMMIT = run_command(
    'resolve source', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR
)

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'torch': '2.13.0',
    'transformers': '4.56.1',
}


def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


requirements = [
    f'{name}=={version}'
    for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if requirements:
    run_command(
        'install pinned dependencies',
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', '--no-cache-dir', *requirements],
    )
run_command(
    'install Diffusion-ARC source',
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--no-input',
        '--no-deps',
        '--no-build-isolation',
        '-e',
        str(SOURCE_DIR),
    ],
)

source_path = str((SOURCE_DIR / 'src').resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
import diffusion_arc

test_environment = os.environ.copy()
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_environment['PYTHONPATH'] = source_path
test_started = time.perf_counter()
test_process = subprocess.run(
    [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=SOURCE_DIR,
    env=test_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
test_seconds = time.perf_counter() - test_started
print('\n'.join(test_process.stdout.splitlines()[-30:]))
if test_process.returncode != 0:
    error = RuntimeError(f'Unit tests returned {test_process.returncode}')
    record_failure('unit-tests', error)
    raise error

REPORT['source'] = {
    'repository': REPOSITORY_URL,
    'requested_revision': SOURCE_REVISION,
    'resolved_commit': RESOLVED_SOURCE_COMMIT,
    'path': str(SOURCE_DIR),
}
REPORT['packages'] = {name: installed_version(name) for name in PINNED_PACKAGES}
REPORT['unit_tests'] = {'returncode': 0, 'seconds': test_seconds}
save_report()
print('Source commit:', RESOLVED_SOURCE_COMMIT)
print(f'Unit tests: PASS in {test_seconds:.2f}s')


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
....................................................................................................................../usr/local/lib/python3.12/site-packages/torch/nn/functional.py:3006: UserWarning: Mismatch dtype between input and weight: input dtype = c10::BFloat16, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/layer_norm.cpp:346.)
  return torch.rms_norm(input, normalized_shape, weight, eps)
................................
----------------------------------------------------------------------
Ran 150 tests in 3.759s

OK
Source commit: 306eb87dae1841669451885102aede283bc22bc4
Unit tests: PASS in 16.57s


## 3. Validate CUDA, model provenance, dataset, and run configuration


In [5]:
import torch

try:
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable; select an H200')
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError('Selected CUDA device does not support BF16')
    DEVICE = torch.device('cuda:0')
    torch.cuda.set_device(DEVICE)
    properties = torch.cuda.get_device_properties(DEVICE)
    gpu_vram_gib = properties.total_memory / 2**30
    if 'H200' not in properties.name.upper() or gpu_vram_gib < MINIMUM_VRAM_GIB:
        raise RuntimeError(
            f'Requires one H200 with >= {MINIMUM_VRAM_GIB:.0f} GiB; '
            f'received {properties.name} with {gpu_vram_gib:.1f} GiB'
        )
except Exception as error:
    record_failure('cuda-contract', error)
    raise


def cuda_memory(label: str) -> dict[str, object]:
    torch.cuda.synchronize(DEVICE)
    return {
        'label': label,
        'allocated_gib': torch.cuda.memory_allocated(DEVICE) / 2**30,
        'reserved_gib': torch.cuda.memory_reserved(DEVICE) / 2**30,
        'peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 2**30,
        'peak_reserved_gib': torch.cuda.max_memory_reserved(DEVICE) / 2**30,
    }


model_manifest = json.loads(MODEL_MANIFEST.read_text(encoding='utf-8'))
if Path(model_manifest['flux']['path']).resolve() != FLUX_CHECKPOINT.resolve():
    raise RuntimeError('FLUX checkpoint path differs from model manifest')
if Path(model_manifest['nvarc']['path']).resolve() != NVARC_CHECKPOINT.resolve():
    raise RuntimeError('NVARC checkpoint path differs from model manifest')

dataset_manifest = json.loads(ARC_DATA_MANIFEST.read_text(encoding='utf-8'))
expected_dataset_source = {
    'repository': 'https://github.com/arcprize/ARC-AGI-2',
    'commit': 'f3283f727488ad98fe575ea6a5ac981e4a188e49',
    'archive_sha256': '179681bc01d55810ee0e90d2717ef04b8d130496541c4853479a8d6fffa4b975',
}
if dataset_manifest.get('source') != expected_dataset_source:
    raise RuntimeError('ARC dataset source identity differs')
if dataset_manifest.get('counts') != {'training': 1000, 'evaluation': 120}:
    raise RuntimeError('ARC dataset task counts differ')
for relative, expected_hash in dataset_manifest['files'].items():
    path = ARC_DATA_ROOT / relative
    if not path.is_file() or sha256_file(path) != expected_hash:
        raise RuntimeError(f'ARC dataset file differs: {relative}')

TEMP_ROOT = Path('/tmp') / f'diffusion-arc-overfit-data-{RUN_ID}'
TEMP_DATA_ROOT = TEMP_ROOT / 'data'
RUNTIME_SPLIT_ROOT = TEMP_DATA_ROOT / 'splits' / 'overfitting_run_1'
RUNTIME_SPLIT_ROOT.parent.mkdir(parents=True)
shutil.copytree(
    SOURCE_DIR / 'data' / 'splits' / 'overfitting_run_1',
    RUNTIME_SPLIT_ROOT,
)
(TEMP_DATA_ROOT / 'raw').mkdir()
os.symlink(ARC_DATA_ROOT, TEMP_DATA_ROOT / 'raw' / 'arc-agi-2', target_is_directory=True)

from diffusion_arc.arc_data import parse_augmentation_key
from diffusion_arc.training.single_gpu_config import load_single_gpu_training_config

source_config_path = SOURCE_DIR / 'configs' / 'training' / 'overfitting' / 'single_gpu_overfitting_pilot.json'
if not source_config_path.is_file():
    raise RuntimeError(
        'The selected source revision lacks single_gpu_overfitting_pilot.json; '
        'push the implementation and select that commit.'
    )
source_config = load_single_gpu_training_config(source_config_path)
RUN_CONFIG = replace(
    source_config,
    models=replace(
        source_config.models,
        flux_checkpoint=FLUX_CHECKPOINT,
        nvarc_checkpoint=NVARC_CHECKPOINT,
    ),
    data=replace(source_config.data, split_directory=RUNTIME_SPLIT_ROOT),
)


def count_view(path: Path) -> tuple[int, int]:
    values = json.loads(path.read_text(encoding='utf-8'))
    targets = sum(
        len(parse_augmentation_key(value).augmentation.pair_indices) - 1
        for value in values
    )
    return len(values), targets


train_counts = count_view(RUNTIME_SPLIT_ROOT / 'augmented_keys' / 'train.json')
unseen_counts = count_view(RUNTIME_SPLIT_ROOT / 'augmented_keys' / 'train_unseen.json')
task_ids = tuple(json.loads((RUNTIME_SPLIT_ROOT / 'train.json').read_text()))
expected_contract = {
    'train_sequences': EXPECTED_TRAIN_SEQUENCES,
    'train_targets': EXPECTED_TRAIN_TARGETS,
    'unseen_sequences': EXPECTED_UNSEEN_SEQUENCES,
    'unseen_targets': EXPECTED_UNSEEN_TARGETS,
    'tasks': EXPECTED_TASKS,
    'steps_per_epoch': STEPS_PER_EPOCH,
}
actual_contract = {
    'train_sequences': train_counts[0],
    'train_targets': train_counts[1],
    'unseen_sequences': unseen_counts[0],
    'unseen_targets': unseen_counts[1],
    'tasks': len(task_ids),
    'steps_per_epoch': train_counts[1] // RUN_CONFIG.loop.optimizer_batch_size,
}
if actual_contract != expected_contract:
    raise RuntimeError(f'Dataset/run contract differs: {actual_contract}')
if RUN_CONFIG.loop.optimizer_batch_size != 8 or RUN_CONFIG.loop.flux_microbatch_size != 1:
    raise RuntimeError('Gradient accumulation contract differs')

REPORT['environment'] = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu_name': properties.name,
    'gpu_vram_gib': gpu_vram_gib,
    'compute_capability': list(torch.cuda.get_device_capability(DEVICE)),
    'bf16_supported': torch.cuda.is_bf16_supported(),
}
REPORT['model_manifest_sha256'] = sha256_file(MODEL_MANIFEST)
REPORT['dataset'] = {
    'manifest_sha256': sha256_file(ARC_DATA_MANIFEST),
    'source': dataset_manifest['source'],
    'counts': dataset_manifest['counts'],
    'run_contract': actual_contract,
    'task_ids': list(task_ids),
    'split_file_sha256': {
        str(path.relative_to(RUNTIME_SPLIT_ROOT)): sha256_file(path)
        for path in sorted(RUNTIME_SPLIT_ROOT.rglob('*.json'))
    },
}
REPORT['resolved_training_config'] = json.loads(
    json.dumps(asdict(RUN_CONFIG), default=str)
)
save_report()
print(actual_contract)
print(f'GPU: {properties.name} | {gpu_vram_gib:.1f} GiB')


{'train_sequences': 512, 'train_targets': 1664, 'unseen_sequences': 128, 'unseen_targets': 416, 'tasks': 8, 'steps_per_epoch': 208}
GPU: NVIDIA H200 | 139.8 GiB


## 4. Load frozen NVARC/VAE and the complete trainable FLUX transformer


In [6]:
from diffusion_arc.training.single_gpu_setup import (
    build_flow_generator,
    build_flux_optimizer,
    load_single_gpu_models,
    prepare_single_gpu_device,
    seed_single_gpu_process,
)

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = prepare_single_gpu_device(RUN_CONFIG)
seed_single_gpu_process(SEED)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

load_started = time.perf_counter()
models = load_single_gpu_models(RUN_CONFIG, device=device)
optimizer = build_flux_optimizer(RUN_CONFIG, transformer=models.transformer)
flow_generator = build_flow_generator(RUN_CONFIG, device=device)
load_seconds = time.perf_counter() - load_started

if any(parameter.requires_grad for parameter in models.nvarc_runner.parameters()):
    raise RuntimeError('NVARC unexpectedly trainable')
if any(parameter.requires_grad for parameter in models.vae_encoder.parameters()):
    raise RuntimeError('VAE unexpectedly trainable')
if not all(parameter.requires_grad for parameter in models.transformer.parameters()):
    raise RuntimeError('Not every FLUX parameter is trainable')

REPORT['models'] = {
    'load_seconds': load_seconds,
    'flux_trainable_parameters': sum(
        parameter.numel() for parameter in models.transformer.parameters()
        if parameter.requires_grad
    ),
    'memory_after_load': cuda_memory('after-model-load'),
}
save_report()
REPORT['models']


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

{'load_seconds': 75.34053215900002,
 'flux_trainable_parameters': 3875544576,
 'memory_after_load': {'label': 'after-model-load',
  'allocated_gib': 21.522281169891357,
  'reserved_gib': 21.564453125,
  'peak_allocated_gib': 21.522281169891357,
  'peak_reserved_gib': 21.564453125}}

## 5. Build leakage-safe canonical evaluation and fixed probes

Canonical NVARC inputs are constructed only from demonstrations and query
inputs. Outputs are loaded separately and enter only VAE scoring latents and
metric computation. One deterministic training view and one unseen view per
task form additional fixed-noise probes.


In [7]:
from torch.utils.data import DataLoader

from diffusion_arc.arc_data import (
    align_evaluation_targets,
    load_evaluation_targets,
    load_evaluation_task,
    render_evaluation_prompts,
    tokenize_nvarc_evaluation_sample,
)
from diffusion_arc.inference import (
    diagnose_exact_grids,
    generate_arc_batch,
    linear_timestep_schedule,
    prepare_evaluation_text_inputs,
)
from diffusion_arc.training import (
    ArcTrainingDataset,
    build_flux_microbatch,
    evaluate_fixed_flow_probe,
    prepare_flow_probe_microbatch,
    prepare_flux_inputs,
    run_single_gpu_training,
)

evaluation_samples = []
expected_grids = []
evaluation_keys = []
for task_id in task_ids:
    task_path = ARC_DATA_ROOT / 'training' / f'{task_id}.json'
    evaluation_task = load_evaluation_task(task_path)
    aligned_targets = align_evaluation_targets(
        evaluation_task,
        load_evaluation_targets(task_path),
    )
    rendered = render_evaluation_prompts(evaluation_task)
    if len(rendered) != 1 or len(aligned_targets) != 1:
        raise RuntimeError(f'Pilot requires one query for task {task_id}')
    evaluation_samples.append(
        tokenize_nvarc_evaluation_sample(
            rendered[0],
            models.tokenizer,
            max_tokens=192,
        )
    )
    expected_grids.append(aligned_targets[0].output_grid)
    evaluation_keys.append((task_id, 0))

canonical_text_inputs = prepare_evaluation_text_inputs(
    evaluation_samples,
    nvarc_runner=models.nvarc_runner,
)
canonical_probe = prepare_flow_probe_microbatch(
    canonical_text_inputs,
    expected_grids,
    vae_encoder=models.vae_encoder,
)


def one_final_target_per_task(view_name: str):
    dataset = ArcTrainingDataset(
        split_directory=RUNTIME_SPLIT_ROOT,
        view_name=view_name,
        tokenizer=models.tokenizer,
    )
    selected = []
    seen = set()
    for index, key in enumerate(dataset.keys):
        if key.task_id in seen:
            continue
        sample = dataset[index]
        inputs = prepare_flux_inputs(
            sample,
            nvarc_runner=models.nvarc_runner,
            vae_encoder=models.vae_encoder,
            min_context_pairs_before_target=len(sample.target_grids),
        )
        if len(inputs) != 1:
            raise RuntimeError(f'Expected one final target for {key}')
        selected.append(inputs[0])
        seen.add(key.task_id)
    if seen != set(task_ids):
        raise RuntimeError(f'{view_name} does not cover every pilot task')
    return build_flux_microbatch(tuple(selected))


train_probe = one_final_target_per_task('train')
unseen_probe = one_final_target_per_task('train_unseen')
PROBE_BATCHES = {
    'train': (train_probe,),
    'unseen_augmentation': (unseen_probe,),
    'canonical': (canonical_probe,),
}

REPORT['evaluation_panel'] = {
    'keys': [list(key) for key in evaluation_keys],
    'target_free_token_counts': [len(sample.input_ids) for sample in evaluation_samples],
    'expected_shapes': [[len(grid), len(grid[0])] for grid in expected_grids],
    'fixed_probe_targets': {
        name: sum(batch.target_count for batch in batches)
        for name, batches in PROBE_BATCHES.items()
    },
}
REPORT['memory_after_probes'] = cuda_memory('after-fixed-probe-preparation')
save_report()
REPORT['evaluation_panel']


{'keys': [['3c9b0459', 0],
  ['0d3d703e', 0],
  ['8be77c9e', 0],
  ['c9e6f938', 0],
  ['00576224', 0],
  ['0520fde7', 0],
  ['88a62173', 0],
  ['d631b094', 0]],
 'target_free_token_counts': [138, 138, 144, 135, 120, 156, 162, 104],
 'expected_shapes': [[3, 3],
  [3, 3],
  [6, 3],
  [3, 6],
  [6, 6],
  [3, 3],
  [2, 2],
  [1, 5]],
 'fixed_probe_targets': {'train': 8, 'unseen_augmentation': 8, 'canonical': 8}}

## 6. Durable probe, generation, and epoch helpers


In [8]:
from PIL import Image

GLOBAL_STEP = 0
COMPLETED_EPOCHS = 0
TIMESTEP_SCHEDULE = linear_timestep_schedule(INFERENCE_STEPS, device=device)


def append_step_record(record: dict[str, object]) -> None:
    with STEPS_PATH.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, sort_keys=True, allow_nan=False) + '\n')
        handle.flush()
        os.fsync(handle.fileno())


def run_fixed_probes(global_step: int) -> dict[str, float]:
    losses = {}
    for name, batches in PROBE_BATCHES.items():
        result = evaluate_fixed_flow_probe(
            models.transformer,
            batches,
            compute_dtype=models.flux_compute_dtype,
            seed=PROBE_SEED,
        )
        losses[name] = result.loss
    entry = {
        'global_step': global_step,
        'losses': losses,
        'memory': cuda_memory(f'probe-step-{global_step}'),
    }
    REPORT['probes'].append(entry)
    save_report()
    print('Fixed probes:', entry)
    return losses


def diagnostic_payload(diagnostic) -> dict[str, object]:
    return {
        'prediction_valid': diagnostic.prediction_valid,
        'invalid_reason': diagnostic.invalid_reason,
        'predicted_shape': diagnostic.predicted_shape,
        'expected_shape': diagnostic.expected_shape,
        'shape_match': diagnostic.shape_match,
        'cell_accuracy': diagnostic.cell_accuracy,
        'exact_grid': diagnostic.exact_grid,
    }


def run_canonical_generation(global_step: int) -> dict[str, object]:
    samples_by_seed = []
    diagnostics_by_seed = []
    for generation_seed in GENERATION_SEEDS:
        result = generate_arc_batch(
            models.transformer,
            models.vae_decoder,
            canonical_text_inputs,
            timestep_schedule=TIMESTEP_SCHEDULE,
            compute_dtype=models.flux_compute_dtype,
            generator=torch.Generator(device=device).manual_seed(generation_seed),
        )
        diagnostics = diagnose_exact_grids(result.predictions, expected_grids)
        seed_samples = []
        for index, ((task_id, query_index), prediction, diagnostic) in enumerate(
            zip(evaluation_keys, result.predictions, diagnostics, strict=True)
        ):
            filename = f'step-{global_step:04d}-{task_id}-q{query_index}-seed-{generation_seed}.png'
            image = result.rgb_images[index].permute(1, 2, 0).contiguous().numpy()
            Image.fromarray(image).save(IMAGES_DIR / filename)
            seed_samples.append({
                'task_id': task_id,
                'query_index': query_index,
                'seed': generation_seed,
                'image': str((IMAGES_DIR / filename).relative_to(REPORT_DIR)),
                'prediction': prediction.grid,
                'diagnostic': diagnostic_payload(diagnostic),
            })
        samples_by_seed.append(seed_samples)
        diagnostics_by_seed.append(diagnostics)

    exact_pass1 = sum(item.exact_grid for item in diagnostics_by_seed[0])
    exact_pass2 = sum(
        first.exact_grid or second.exact_grid
        for first, second in zip(*diagnostics_by_seed, strict=True)
    )
    valid_pass2 = sum(
        first.prediction_valid or second.prediction_valid
        for first, second in zip(*diagnostics_by_seed, strict=True)
    )
    shape_pass2 = sum(
        first.shape_match or second.shape_match
        for first, second in zip(*diagnostics_by_seed, strict=True)
    )
    best_cell_accuracies = []
    for first, second in zip(*diagnostics_by_seed, strict=True):
        available = [
            value for value in (first.cell_accuracy, second.cell_accuracy)
            if value is not None
        ]
        best_cell_accuracies.append(max(available) if available else None)

    entry = {
        'global_step': global_step,
        'exact_grid_pass1': exact_pass1,
        'exact_grid_pass2': exact_pass2,
        'valid_grid_pass2': valid_pass2,
        'correct_shape_pass2': shape_pass2,
        'best_cell_accuracies': best_cell_accuracies,
        'samples_by_seed': samples_by_seed,
        'memory': cuda_memory(f'generation-step-{global_step}'),
    }
    REPORT['generation'].append(entry)
    save_report()
    print({key: value for key, value in entry.items() if key != 'samples_by_seed'})
    return entry


def evidence_summary() -> dict[str, object]:
    if not REPORT['probes'] or not REPORT['generation']:
        raise RuntimeError('Baseline probes and generation are missing')
    baseline_probe = REPORT['probes'][0]['losses']['train']
    final_probe = REPORT['probes'][-1]['losses']['train']
    final_generation = REPORT['generation'][-1]
    baseline_cells = REPORT['generation'][0]['best_cell_accuracies']
    final_cells = final_generation['best_cell_accuracies']

    def available_mean(values):
        present = [value for value in values if value is not None]
        return None if not present else sum(present) / len(present)

    baseline_cell_mean = available_mean(baseline_cells)
    final_cell_mean = available_mean(final_cells)
    cell_improved = (
        final_cell_mean is not None
        and (baseline_cell_mean is None or final_cell_mean > baseline_cell_mean)
    )
    summary = {
        'train_probe_loss_ratio': final_probe / baseline_probe,
        'optimization_signal': final_probe / baseline_probe <= 0.5,
        'structural_signal': (
            final_generation['correct_shape_pass2'] >= 4 and cell_improved
        ),
        'positive_overfitting_evidence': final_generation['exact_grid_pass2'] >= 1,
        'strong_overfitting_evidence': final_generation['exact_grid_pass2'] >= 6,
        'complete_overfitting': final_generation['exact_grid_pass2'] == 8,
        'baseline_conditional_cell_accuracy': baseline_cell_mean,
        'final_conditional_cell_accuracy': final_cell_mean,
    }
    REPORT['evidence_summary'] = summary
    save_report()
    return summary


def run_epoch(epoch: int) -> dict[str, object]:
    global GLOBAL_STEP, COMPLETED_EPOCHS
    if epoch != COMPLETED_EPOCHS + 1 or not 1 <= epoch <= EPOCH_COUNT:
        raise RuntimeError(
            f'Next epoch must be {COMPLETED_EPOCHS + 1}, received {epoch}'
        )

    epoch_config = replace(RUN_CONFIG, seed=SEED + epoch - 1)
    dataset = ArcTrainingDataset(
        split_directory=RUNTIME_SPLIT_ROOT,
        view_name='train',
        tokenizer=models.tokenizer,
    )
    loader = DataLoader(
        dataset,
        batch_size=None,
        shuffle=True,
        num_workers=0,
        generator=torch.Generator(device='cpu').manual_seed(epoch_config.seed),
    )
    epoch_records = []
    epoch_started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(device)

    def on_step(metrics) -> None:
        nonlocal epoch_records
        global GLOBAL_STEP
        GLOBAL_STEP += 1
        record = {
            'epoch': epoch,
            'epoch_step': len(epoch_records) + 1,
            'global_step': GLOBAL_STEP,
            'loss': float(metrics.loss.item()),
            'target_count': metrics.target_count,
            'microbatch_count': metrics.microbatch_count,
            'gradient_norm': metrics.gradient_norm,
            'gradient_clipped': metrics.gradient_clipped,
            'learning_rates': list(metrics.learning_rates),
            'optimizer_step_seconds': metrics.step_seconds,
            'cuda_allocated_gib': torch.cuda.memory_allocated(device) / 2**30,
            'cuda_reserved_gib': torch.cuda.memory_reserved(device) / 2**30,
            'cuda_peak_allocated_gib': torch.cuda.max_memory_allocated(device) / 2**30,
        }
        epoch_records.append(record)
        append_step_record(record)
        if GLOBAL_STEP % PROBE_INTERVAL_STEPS == 0:
            run_fixed_probes(GLOBAL_STEP)

    try:
        run_single_gpu_training(
            loader,
            nvarc_runner=models.nvarc_runner,
            vae_encoder=models.vae_encoder,
            transformer=models.transformer,
            optimizer=optimizer,
            flux_compute_dtype=models.flux_compute_dtype,
            max_optimizer_steps=STEPS_PER_EPOCH,
            optimizer_batch_size=RUN_CONFIG.loop.optimizer_batch_size,
            flux_microbatch_size=RUN_CONFIG.loop.flux_microbatch_size,
            min_context_pairs_before_target=RUN_CONFIG.loop.min_context_pairs_before_target,
            generator=flow_generator,
            max_grad_norm=MAX_GRAD_NORM,
            on_optimizer_step_metrics=on_step,
        )
    except Exception as error:
        record_failure(f'epoch-{epoch}', error)
        raise

    if len(epoch_records) != STEPS_PER_EPOCH:
        error = RuntimeError(
            f'Epoch {epoch} completed {len(epoch_records)} steps, expected {STEPS_PER_EPOCH}'
        )
        record_failure(f'epoch-{epoch}-length', error)
        raise error

    epoch_seconds = time.perf_counter() - epoch_started
    COMPLETED_EPOCHS = epoch
    generation = run_canonical_generation(GLOBAL_STEP)
    epoch_summary = {
        'epoch': epoch,
        'first_global_step': epoch_records[0]['global_step'],
        'last_global_step': epoch_records[-1]['global_step'],
        'seconds': epoch_seconds,
        'mean_training_loss': sum(record['loss'] for record in epoch_records) / len(epoch_records),
        'mean_gradient_norm': sum(record['gradient_norm'] for record in epoch_records) / len(epoch_records),
        'clipped_updates': sum(record['gradient_clipped'] for record in epoch_records),
        'memory': cuda_memory(f'epoch-{epoch}-complete'),
        'exact_grid_pass2': generation['exact_grid_pass2'],
    }
    REPORT['epochs'].append(epoch_summary)
    REPORT['completed_epochs'] = COMPLETED_EPOCHS
    REPORT['completed_optimizer_steps'] = GLOBAL_STEP
    REPORT['status'] = 'COMPLETE' if epoch == EPOCH_COUNT else 'PARTIAL'
    REPORT['evidence_summary'] = evidence_summary()
    save_report()
    return epoch_summary


## 7. Step-0 baselines

Run once before any optimizer update.


In [9]:
if GLOBAL_STEP != 0 or REPORT['probes'] or REPORT['generation']:
    raise RuntimeError('Step-0 baseline has already been recorded')
REPORT['status'] = 'BASELINE_COMPLETE'
baseline_probe_losses = run_fixed_probes(0)
baseline_generation = run_canonical_generation(0)
save_report()
{'probe_losses': baseline_probe_losses, 'generation': {
    key: value for key, value in baseline_generation.items() if key != 'samples_by_seed'
}}


Fixed probes: {'global_step': 0, 'losses': {'train': 0.17930014431476593, 'unseen_augmentation': 0.1717759072780609, 'canonical': 0.16840004920959473}, 'memory': {'label': 'probe-step-0', 'allocated_gib': 21.619651317596436, 'reserved_gib': 32.150390625, 'peak_allocated_gib': 30.377772331237793, 'peak_reserved_gib': 32.150390625}}
{'global_step': 0, 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 1, 'correct_shape_pass2': 0, 'best_cell_accuracies': [None, None, None, None, None, None, None, None], 'memory': {'label': 'generation-step-0', 'allocated_gib': 21.619651317596436, 'reserved_gib': 36.546875, 'peak_allocated_gib': 30.377772331237793, 'peak_reserved_gib': 36.546875}}


{'probe_losses': {'train': 0.17930014431476593,
  'unseen_augmentation': 0.1717759072780609,
  'canonical': 0.16840004920959473},
 'generation': {'global_step': 0,
  'exact_grid_pass1': 0,
  'exact_grid_pass2': 0,
  'valid_grid_pass2': 1,
  'correct_shape_pass2': 0,
  'best_cell_accuracies': [None, None, None, None, None, None, None, None],
  'memory': {'label': 'generation-step-0',
   'allocated_gib': 21.619651317596436,
   'reserved_gib': 36.546875,
   'peak_allocated_gib': 30.377772331237793,
   'peak_reserved_gib': 36.546875}}}

## 8. Epoch 1 — manually start steps 1–208


In [10]:
epoch_1 = run_epoch(1)
epoch_1


Fixed probes: {'global_step': 52, 'losses': {'train': 0.0338563472032547, 'unseen_augmentation': 0.03412947058677673, 'canonical': 0.03479244187474251}, 'memory': {'label': 'probe-step-52', 'allocated_gib': 64.97550630569458, 'reserved_gib': 78.91015625, 'peak_allocated_gib': 76.96989393234253, 'peak_reserved_gib': 78.91015625}}
Fixed probes: {'global_step': 104, 'losses': {'train': 0.027908537536859512, 'unseen_augmentation': 0.027889905497431755, 'canonical': 0.028010636568069458}, 'memory': {'label': 'probe-step-104', 'allocated_gib': 64.97588777542114, 'reserved_gib': 78.912109375, 'peak_allocated_gib': 76.97443723678589, 'peak_reserved_gib': 78.912109375}}
Fixed probes: {'global_step': 156, 'losses': {'train': 0.021399620920419693, 'unseen_augmentation': 0.021481219679117203, 'canonical': 0.022709054872393608}, 'memory': {'label': 'probe-step-156', 'allocated_gib': 64.97644710540771, 'reserved_gib': 78.912109375, 'peak_allocated_gib': 76.97443723678589, 'peak_reserved_gib': 78.912

{'epoch': 1,
 'first_global_step': 1,
 'last_global_step': 208,
 'seconds': 233.905961837,
 'mean_training_loss': 0.03871296066790819,
 'mean_gradient_norm': 2.2602470729929895,
 'clipped_updates': 135,
 'memory': {'label': 'epoch-1-complete',
  'allocated_gib': 64.96669912338257,
  'reserved_gib': 81.55078125,
  'peak_allocated_gib': 76.97443723678589,
  'peak_reserved_gib': 81.55078125},
 'exact_grid_pass2': 0}

## 9. Epoch 2 — manually start steps 209–416


In [11]:
epoch_2 = run_epoch(2)
epoch_2


Fixed probes: {'global_step': 260, 'losses': {'train': 0.021220097318291664, 'unseen_augmentation': 0.02117741107940674, 'canonical': 0.021987309679389}, 'memory': {'label': 'probe-step-260', 'allocated_gib': 64.97661781311035, 'reserved_gib': 81.55078125, 'peak_allocated_gib': 76.96728849411011, 'peak_reserved_gib': 81.55078125}}
Fixed probes: {'global_step': 312, 'losses': {'train': 0.02608281932771206, 'unseen_augmentation': 0.025988001376390457, 'canonical': 0.027442386373877525}, 'memory': {'label': 'probe-step-312', 'allocated_gib': 64.97442626953125, 'reserved_gib': 81.55078125, 'peak_allocated_gib': 76.96945524215698, 'peak_reserved_gib': 81.55078125}}
Fixed probes: {'global_step': 364, 'losses': {'train': 0.018355755135416985, 'unseen_augmentation': 0.0186839010566473, 'canonical': 0.01990382745862007}, 'memory': {'label': 'probe-step-364', 'allocated_gib': 64.97722482681274, 'reserved_gib': 81.55078125, 'peak_allocated_gib': 76.96952962875366, 'peak_reserved_gib': 81.55078125

{'epoch': 2,
 'first_global_step': 209,
 'last_global_step': 416,
 'seconds': 220.17745788399998,
 'mean_training_loss': 0.02967388746821178,
 'mean_gradient_norm': 1.1310020428007612,
 'clipped_updates': 85,
 'memory': {'label': 'epoch-2-complete',
  'allocated_gib': 64.96609258651733,
  'reserved_gib': 81.55078125,
  'peak_allocated_gib': 76.96952962875366,
  'peak_reserved_gib': 81.55078125},
 'exact_grid_pass2': 0}

## 10. Epoch 3 — manually start steps 417–624


In [12]:
epoch_3 = run_epoch(3)
epoch_3


Fixed probes: {'global_step': 468, 'losses': {'train': 0.01964653842151165, 'unseen_augmentation': 0.019289884716272354, 'canonical': 0.020588383078575134}, 'memory': {'label': 'probe-step-468', 'allocated_gib': 64.9756669998169, 'reserved_gib': 81.55078125, 'peak_allocated_gib': 76.97061491012573, 'peak_reserved_gib': 81.55078125}}
Fixed probes: {'global_step': 520, 'losses': {'train': 0.017696218565106392, 'unseen_augmentation': 0.01743665337562561, 'canonical': 0.020180588588118553}, 'memory': {'label': 'probe-step-520', 'allocated_gib': 64.97670984268188, 'reserved_gib': 81.55078125, 'peak_allocated_gib': 76.97061491012573, 'peak_reserved_gib': 81.55078125}}
Fixed probes: {'global_step': 572, 'losses': {'train': 0.0144259138032794, 'unseen_augmentation': 0.014217411167919636, 'canonical': 0.01620395854115486}, 'memory': {'label': 'probe-step-572', 'allocated_gib': 64.97662496566772, 'reserved_gib': 81.55078125, 'peak_allocated_gib': 76.97061491012573, 'peak_reserved_gib': 81.550781

{'epoch': 3,
 'first_global_step': 417,
 'last_global_step': 624,
 'seconds': 219.96401242000002,
 'mean_training_loss': 0.022210471900717284,
 'mean_gradient_norm': 0.8221654283742492,
 'clipped_updates': 57,
 'memory': {'label': 'epoch-3-complete',
  'allocated_gib': 64.96693563461304,
  'reserved_gib': 81.55078125,
  'peak_allocated_gib': 76.97061491012573,
  'peak_reserved_gib': 81.55078125},
 'exact_grid_pass2': 0}

## 11. Epoch 4 — manually start steps 625–832


In [13]:
epoch_4 = run_epoch(4)
epoch_4


Fixed probes: {'global_step': 676, 'losses': {'train': 0.018599674105644226, 'unseen_augmentation': 0.018436266109347343, 'canonical': 0.019121890887618065}, 'memory': {'label': 'probe-step-676', 'allocated_gib': 64.97592687606812, 'reserved_gib': 81.55078125, 'peak_allocated_gib': 76.96937131881714, 'peak_reserved_gib': 81.55078125}}
Fixed probes: {'global_step': 728, 'losses': {'train': 0.0185101181268692, 'unseen_augmentation': 0.018679356202483177, 'canonical': 0.01907472498714924}, 'memory': {'label': 'probe-step-728', 'allocated_gib': 64.97568321228027, 'reserved_gib': 81.55078125, 'peak_allocated_gib': 76.97114133834839, 'peak_reserved_gib': 81.55078125}}
Fixed probes: {'global_step': 780, 'losses': {'train': 0.013357346877455711, 'unseen_augmentation': 0.013837670907378197, 'canonical': 0.015090041793882847}, 'memory': {'label': 'probe-step-780', 'allocated_gib': 64.97538614273071, 'reserved_gib': 81.55078125, 'peak_allocated_gib': 76.97114133834839, 'peak_reserved_gib': 81.550

{'epoch': 4,
 'first_global_step': 625,
 'last_global_step': 832,
 'seconds': 212.9254059729999,
 'mean_training_loss': 0.02118205136950844,
 'mean_gradient_norm': 0.6852373327128589,
 'clipped_updates': 34,
 'memory': {'label': 'epoch-4-complete',
  'allocated_gib': 64.96669912338257,
  'reserved_gib': 81.55078125,
  'peak_allocated_gib': 76.97114133834839,
  'peak_reserved_gib': 81.55078125},
 'exact_grid_pass2': 0}

## 12. Final evidence summary

This cell is read-only with respect to model weights. It summarizes the latest
completed epoch; partial runs remain explicitly labelled `PARTIAL`.


In [14]:
summary = evidence_summary()
print('Status:', REPORT['status'])
print('Completed epochs:', COMPLETED_EPOCHS)
print('Completed optimizer steps:', GLOBAL_STEP)
print('Report:', REPORT_PATH)
summary


Status: COMPLETE
Completed epochs: 4
Completed optimizer steps: 832
Report: /mnt/diffusion-arc-models/results/single-gpu-overfitting-pilot/20260825T060523831464Z/report.json


{'train_probe_loss_ratio': 0.08645645415478197,
 'optimization_signal': True,
 'structural_signal': False,
 'positive_overfitting_evidence': False,
 'strong_overfitting_evidence': False,
 'complete_overfitting': False,
 'baseline_conditional_cell_accuracy': None,
 'final_conditional_cell_accuracy': 0.0555555559694767}

## 13. Optional explicit checkpoint save

Run this cell only if you intentionally want to persist the current FP32 FLUX
transformer. It writes a large checkpoint to this run's durable result directory;
optimizer/resume state is not included in this pilot.


In [15]:
CHECKPOINT_DIR = REPORT_DIR / f'flux-transformer-step-{GLOBAL_STEP:04d}'
if GLOBAL_STEP == 0:
    raise RuntimeError('Refusing to save the untrained step-0 transformer')
if CHECKPOINT_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite checkpoint: {CHECKPOINT_DIR}')
models.transformer.save_pretrained(CHECKPOINT_DIR, safe_serialization=True)
REPORT['saved_checkpoint'] = {
    'global_step': GLOBAL_STEP,
    'path': str(CHECKPOINT_DIR),
    'files': {
        str(path.relative_to(CHECKPOINT_DIR)): {
            'bytes': path.stat().st_size,
            'sha256': sha256_file(path),
        }
        for path in sorted(CHECKPOINT_DIR.rglob('*'))
        if path.is_file()
    },
}
save_report()
REPORT['saved_checkpoint']


{'global_step': 832,
 'path': '/mnt/diffusion-arc-models/results/single-gpu-overfitting-pilot/20260825T060523831464Z/flux-transformer-step-0832',
 'files': {'config.json': {'bytes': 530,
   'sha256': '120ac9765fb702575e823221b9869c6de6f80adb6cf76b131842c9e5ef109c7c'},
  'diffusion_pytorch_model-00001-of-00002.safetensors': {'bytes': 9876047152,
   'sha256': '063217615fec57f609360caa4c80094b7fa09ea9ce2e7943b4a7a68ca65adaa3'},
  'diffusion_pytorch_model-00002-of-00002.safetensors': {'bytes': 5626151600,
   'sha256': 'd7719954578b35075603759303420aa5dd4ba273279ad432dc79726acd9f7c8a'},
  'diffusion_pytorch_model.safetensors.index.json': {'bytes': 18109,
   'sha256': 'd7fd9ef8cfa0b60d804f043f0660c21e23c7d8787f7301c45d804f88479fc6c1'}}}